**Nodes (Entities)**

- People (employees, affiliates) are represented as nodes, labeled with their names.
- Companies and Organizations are also nodes, distinguished by different colors.

**Edges (Connections)**

- If a person worked at a company, there’s an edge between the person node and the company node.
- If two people worked in the same organization at the same time, an edge connects them.
- If two people share a company, an edge is created between them as well.

**Node Attributes**

- Color-coded: Each node’s color depends on its entity type (e.g., people, companies, organizations).
- Size-based on connections: More connected nodes (popular people or big companies) appear larger.

**Interactivity**

- Hovering over a node shows details like job title, company name, and years of employment.
- Clicking a node can reveal more about its connections.

**What This Graph Helps With**
- Identifying key influencers: People with many connections (e.g., well-networked employees).
- Tracing employment history: See where a person worked and their colleagues.
- Understanding organizational structures: Which companies or organizations have the most connections.


In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import random
from IPython.core.display import display, HTML

In [ ]:
# Load data
ipc = pd.read_excel('IndividualPrimaryContact.xlsx')
ieh = pd.read_excel('IndividualEmploymentHistory.xlsx')
cpc = pd.read_excel('CompanyPrimaryContact.xlsx')
iah = pd.read_excel('IndividualAffiliationHistory.xlsx')

In [ ]:
# Generate a color for each company
def get_unique_colors(categories):
    """Assign unique colors to companies/organizations."""
    unique_categories = list(set(categories.dropna()))
    color_map = {category: f"#{random.randint(0, 0xFFFFFF):06x}" for category in unique_categories}
    return color_map

# Function to build graph from data
def build_graph(ipc, ieh, iah, color_map):
    """Creates a network graph with colored and sized nodes."""
    G = nx.Graph()
    
    # Add individuals
    for row in ipc.itertuples():
        name = f"{row.FirstName} {row.LastName}"
        company = row.Company
        color = color_map.get(company, "#CCCCCC")  # Default gray if company not found
        G.add_node(name, 
                   label=name, 
                   title=f"Name:{row.Label} \n PID:{row.PID} \n Job:{row.Job} \n Company:{company}", 
                   color=color)
    
    # Add employment connections
    for row in ieh.itertuples():
        person_name = f"{ipc.loc[ipc['PID'] == row.PID, 'FirstName'].values[0]} {ipc.loc[ipc['PID'] == row.PID, 'LastName'].values[0]}"
        G.add_node(row.Company, label=row.Company, color=color_map.get(row.Company, "#CCCCCC"))
        G.add_edge(person_name, row.Company, title=f"Worked as {row.Role} from {row._5} to {row._6}")

    # Add affiliation connections
    for row in iah.itertuples():
        person_name = f"{ipc.loc[ipc['PID'] == row.PID, 'FirstName'].values[0]} {ipc.loc[ipc['PID'] == row.PID, 'LastName'].values[0]}"
        G.add_node(row.Organization, label=row.Organization, color=color_map.get(row.Organization, "#CCCCCC"))
        G.add_edge(person_name, row.Organization, title=f"Role: {row.Role} ({row._5} - {row._6})")

    return G

# Function to adjust node sizes based on connections
def adjust_node_sizes(G):
    """Sets node size based on the number of connections (degree centrality)."""
    degrees = dict(G.degree())
    nx.set_node_attributes(G, {node: max(10, degree * 3) for node, degree in degrees.items()}, "size")

# Function to visualize graph
def visualize_graph(G, filename="network.html"):
    """Visualizes the graph using Pyvis."""
    net = Network(notebook=True, height="750px", width="100%", bgcolor="#222222", font_color="white", cdn_resources="in_line")
    net.from_nx(G)
    net.show(filename)

In [ ]:
# Assign colors to companies and organizations
company_colors = get_unique_colors(ipc["Company"])
organization_colors = get_unique_colors(iah["Organization"])
color_map = {**company_colors, **organization_colors}

# Build the graph
G = build_graph(ipc, ieh, iah, color_map)

# Adjust node sizes
adjust_node_sizes(G)

# Visualize
# visualize_graph(G)                       # Not working on JupyterNB so extracted the function outside

In [ ]:
net = Network(notebook=True, height="750px", width="100%", bgcolor="#222222", font_color="white", cdn_resources="in_line")
net.from_nx(G)
net.show('network.html')

In [ ]:
def get_overlap(start1, end1, start2, end2):
    """Check if two time periods overlap."""
    return max(start1, start2) <= min(end1, end2)

def get_unique_colors(categories):
    """Assign unique colors to companies/organizations."""
    unique_categories = list(set(categories.dropna()))
    color_map = {category: f"#{random.randint(0, 0xFFFFFF):06x}" for category in unique_categories}
    return color_map

def get_subgraph_by_pid(pid, ipc, iah):
    # Create a NetworkX graph
    G = nx.Graph()

    # Get details of the input person
    person_info = ipc[ipc['PID'] == pid]
    
    if person_info.empty:
        print(f"No information found for PID: {pid}")
        return None

    person_name = person_info.iloc[0]['FirstName'] + ' ' + person_info.iloc[0]['LastName']
    person_job = person_info.iloc[0]['Job'] if 'Job' in person_info else "Unknown"
    person_company = person_info.iloc[0]['Company'] if 'Company' in person_info else "Unknown"

    # Get the list of organizations and years the person worked at
    person_affiliations = iah[iah['PID'] == pid]

    if person_affiliations.empty:
        print(f"No affiliations found for {person_name}.")
        return None

    # Generate colors for organizations
    organization_colors = get_unique_colors(iah["Organization"])

    # Add the main person to the graph
    G.add_node(person_name, 
               label=person_name, 
               title=f"PID: {pid} \nJob: {person_job} \nCompany: {person_company}", 
               color='blue')

    # Dictionary to store organization members
    organizations = {}

    # Group people by organizations
    for row in iah.itertuples():
        org = row.Organization
        if org not in organizations:
            organizations[org] = []
        organizations[org].append((row.PID, row.Role, row._5, row._6))  # (PID, Role, Start, End)

    # Dictionary to store connected people (PID -> Full Name)
    connected_people = {}

    # Find people who worked in the same organizations at overlapping times
    for org, org_members in organizations.items():
        # Add organization node
        G.add_node(org, label=org, color=organization_colors.get(org, "#CCCCCC"))  # Default gray color
        
        for person1 in org_members:
            pid1, role1, start1, end1 = person1
            if pid1 != pid:
                continue  # Skip self

            # Link main person to organization
            G.add_edge(person_name, org, title=f"Role: {role1} ({start1} - {end1})")

            for person2 in org_members:
                pid2, role2, start2, end2 = person2
                if pid1 == pid2:
                    continue  # Skip self-pairs

                if get_overlap(start1, end1, start2, end2):
                    if pid2 not in connected_people:
                        other_person_info = ipc[ipc['PID'] == pid2]
                        if not other_person_info.empty:
                            other_name = other_person_info.iloc[0]['FirstName'] + ' ' + other_person_info.iloc[0]['LastName']
                            other_job = other_person_info.iloc[0]['Job'] if 'Job' in other_person_info else "Unknown"
                            other_company = other_person_info.iloc[0]['Company'] if 'Company' in other_person_info else "Unknown"
                            connected_people[other_name] = (pid2, role2, org, start2, end2, other_job, other_company)

    # Add the connected people to the graph using names (not PIDs)
    for other_name, (pid2, role, org, start, end, job, company) in connected_people.items():
        G.add_node(other_name, 
                   label=other_name, 
                   title=f"PID: {pid2} \nJob: {job} \nCompany: {company}", 
                   color='green')
        G.add_edge(other_name, org, title=f"Role: {role} ({start} - {end})")

    # Extract subgraph with correct names
    subgraph_nodes = [person_name] + list(connected_people.keys()) + list(organizations.keys())
    subgraph = G.subgraph(subgraph_nodes).copy()

    # Assign node sizes based on degree (number of connections)
    node_sizes = {node: max(10, G.degree(node) * 5) for node in G.nodes()}  # Ensure minimum size of 10

    print(f"Main Person: {person_name}")
    
    # Create a Pyvis network object and display the subgraph
    net = Network(notebook=True, cdn_resources='in_line')
    net.from_nx(subgraph)

    # Set the size of each node dynamically
    for node in net.nodes:
        node["size"] = node_sizes[node["id"]]  # Assign the calculated size based on connections

    file_name = f"{pid}_subgraph.html"
    net.show(file_name)

    # Display the network graph in Jupyter
    display(HTML(file_name))

    return subgraph

In [ ]:
subgraph = get_subgraph_by_pid('PID_Wy-6p3R', ipc, iah)